# Terafab conditional production-to-energy security study

Reproducible Step 5 results workflow for the prospective 2026–2050 scenario analysis. This source-available research notebook contains no verified Terafab operating data and makes no official feasibility, schedule, engineering, regulatory, investment, endorsement, or affiliation claim. The public `1 TW/year` phrase is modeled conditionally as annual rated device output; it is never treated as a 1 TW facility electrical load.

## Reproducibility mode

Use `quick` while editing. Change `MODE` to `final` for the locked 16,384 → 32,768 Sobol convergence test and 1,000 sensitivity bootstrap replicates. All evidence is loaded from frozen local snapshots; this notebook performs no live web retrieval.

In [ ]:
import csv
import json
from pathlib import Path
from pprint import pprint

from terafab_energy_security.evidence import FrozenEvidence
from terafab_energy_security.exports import write_json, write_scenario_outputs
from terafab_energy_security.pathways import load_scenario_config, run_scenario_matrix
from terafab_energy_security.publication import build_publication_bundle
from terafab_energy_security.uncertainty import (
    correlation_stress_test, dependence_aware_sensitivity,
    independent_sobol_sensitivity, load_uncertainty_contract,
)

MODE = "quick"  # change to "final" for publication calculations
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "studies" / "ecm_terafab_energy_security").exists():
    raise RuntimeError("Run this notebook from the repository root after installing .[study].")
STUDY_ROOT = REPO_ROOT / "studies" / "ecm_terafab_energy_security"
OUTPUT_ROOT = STUDY_ROOT / "outputs" / MODE


## Frozen inputs and scenario contract

The two undisclosed production scales remain numerically indeterminate. The alternative public target meaning also remains explicitly unresolved.

In [ ]:
evidence = FrozenEvidence.load(STUDY_ROOT)
config = load_scenario_config(STUDY_ROOT / "scenarios" / "scenario_matrix.json")
uncertainty_contract = load_uncertainty_contract(STUDY_ROOT / "evidence" / "uncertainty_dependencies.json")
pprint({
    "frozen_sources": len(evidence.sources),
    "frozen_parameters": len(evidence.parameters),
    "horizon": [config["start_year"], config["end_year"]],
    "alternative_target_semantics": config["target_semantic_branches"]["other_publicly_undefined_meaning"],
})


## Complete publication-results bundle

This cell executes the no-build counterfactual and all predeclared target-scale, realization, supply-portfolio, and stress combinations, then creates every prespecified figure, table, data file, and checksum manifest. It uses the same public bundle function as the command-line tool.

In [ ]:
publication_manifest = build_publication_bundle(
    STUDY_ROOT, OUTPUT_ROOT / "publication", mode=MODE
)
pprint(publication_manifest["output_integrity"])
pprint(publication_manifest["scenario_manifest"])
print(publication_manifest["manifest_path"])


In [ ]:
representative_ids = {
    "full_announced_target__accelerated__grid_dominant__normal",
    "full_announced_target__accelerated__firm_onsite__infrastructure_delay",
    "full_announced_target__accelerated__grid_dominant__cooling_or_water_constraint",
}
annual_path = OUTPUT_ROOT / "publication" / "raw" / "scenarios" / "annual_results.csv"
with annual_path.open(encoding="utf-8") as stream:
    representative_2050 = [
        row for row in csv.DictReader(stream)
        if row["scenario_id"] in representative_ids and row["year"] == "2050"
    ]
for row in representative_2050:
    pprint({key: row.get(key) for key in (
        "scenario_id", "wafer_starts_per_year", "annual_electricity_MWh",
        "coincident_peak_load_MW", "external_water_withdrawal_m3",
        "classification", "binding_constraint", "regional_adequacy_status",
    )})


## Locked uncertainty, dependence, and convergence checks

Feasibility probability is reported as unavailable—not zero—because public manufacturing/facility capacities and official probabilistic ERCOT series needed by the required gates are absent. Resource-demand uncertainty remains estimable.

In [ ]:
uncertainty_path = OUTPUT_ROOT / "publication" / "raw" / "uncertainty_report.json"
uncertainty_report = json.loads(uncertainty_path.read_text(encoding="utf-8"))
correlation_report = uncertainty_report["correlation_stress"]
sobol_report = uncertainty_report["independent_sobol"]
shapley_report = uncertainty_report["dependence_aware"]
pprint([{"dependence": item["dependence"], "shift": item["correlation_shift"],
         "pass": item["estimable_outputs_pass"], "changes": item["relative_changes"]}
        for item in correlation_report])


## Predeclared hypothesis disposition guard

The results bundle must not turn a screening reserve margin into an official adequacy finding. H1–H3 remain indeterminate without the missing gate evidence. H4 also remains indeterminate because resource-peak sensitivity cannot replace its locked realization-year or joint-feasibility outcome. Step 6 may write the paper around conditional resource envelopes and binding unknowns, with these limitations prominent.

In [ ]:
hypothesis_disposition = {
    "H1": "indeterminate_missing_joint_gate_evidence",
    "H2": "indeterminate_missing_official_probabilistic_ercot_metrics",
    "H3": "indeterminate_missing_hourly_adequacy_or_accredited_capacity_evidence",
    "H4": sobol_report["H4_status"],
}
write_json(OUTPUT_ROOT / "hypothesis_disposition.json", hypothesis_disposition)
pprint(hypothesis_disposition)
